# **Exploratory data analysis on the job market as of 2025**

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
jobs = pd.read_csv('../data/jobs25.csv')

In [3]:
jobs.head()

jobs.info()

jobs[['salary_min', 'salary_max', 'salary_mid']].describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 20 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   id                   4000 non-null   int64  
 1   title                4000 non-null   object 
 2   company_name         3786 non-null   object 
 3   category_label       4000 non-null   object 
 4   category_tag         4000 non-null   object 
 5   country              4000 non-null   object 
 6   location_display     4000 non-null   object 
 7   location_area        4000 non-null   object 
 8   latitude             3257 non-null   float64
 9   longitude            3257 non-null   float64
 10  contract_type        846 non-null    object 
 11  contract_time        985 non-null    object 
 12  salary_min           1544 non-null   float64
 13  salary_max           1535 non-null   float64
 14  salary_mid           1544 non-null   float64
 15  salary_is_predicted  4000 non-null   i

,salary_min,salary_max,salary_mid
count,1.544000e+03,1.535000e+03,1.544000e+03
mean,4.549448e+04,5.627804e+04,5.080285e+04
std,9.365883e+04,2.223390e+05,1.567656e+05
min,0.000000e+00,1.000000e+00,1.000000e+00
25%,2.200000e+04,2.543800e+04,2.405000e+04
50%,3.461617e+04,4.000000e+04,3.750000e+04
75%,5.530624e+04,6.156134e+04,5.750000e+04
max,2.000000e+06,5.000000e+06,3.500000e+06


## Data Overview

The dataset contains 4,000 job postings from the UK market in 2025. Only about 38% of jobs have salary information, with mid salaries ranging widely from £1 to £3.5 million.

In [4]:
category_counts = jobs['category_label'].value_counts().head(10)
fig = px.bar(category_counts, title='Top 10 Job Categories')
fig.show()

## Job Categories

Teaching and healthcare jobs dominate the listings, followed by IT and logistics roles.

In [5]:
salary_data = jobs.dropna(subset=['salary_mid'])
fig = px.histogram(salary_data, x='salary_mid', nbins=50, title='Salary Distribution')
fig.show()

## Salary Distribution

Most salaries cluster around £30,000-£50,000, with a long tail of high-paying positions.

In [6]:
avg_salary_by_category = salary_data.groupby('category_label')['salary_mid'].mean().sort_values(ascending=False).head(10)
fig = px.bar(avg_salary_by_category, title='Average Salary by Category')
fig.show()

## Average Salary by Category

Energy and legal jobs offer the highest average salaries, while hospitality and retail have lower averages.

In [7]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_percentage_error
import joblib

# Prepare data
jobs_salary = jobs.dropna(subset=['salary_mid'])
le_category = LabelEncoder()
jobs_salary.loc[:, 'category_encoded'] = le_category.fit_transform(jobs_salary['category_label'])
jobs_salary.loc[:, 'location_simple'] = jobs_salary['location_display'].str.split(',').str[0]
le_location = LabelEncoder()
jobs_salary.loc[:, 'location_encoded'] = le_location.fit_transform(jobs_salary['location_simple'])

X = jobs_salary[['category_encoded', 'location_encoded']]
y = jobs_salary['salary_mid']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
mape = mean_absolute_percentage_error(y_test, y_pred)

print(f'MAPE: {mape:.1f}%')

# Save model
joblib.dump(model, '../models/salary_predictor.joblib')
joblib.dump(le_category, '../models/category_encoder.joblib')
joblib.dump(le_location, '../models/location_encoder.joblib')

/tmp/ipykernel_371886/3837866590.py:10: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ipykernel_371886/3837866590.py:11: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ipykernel_371886/3837866590.py:13: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



MAPE: 45.2%


['../models/location_encoder.joblib']

## Salary Prediction Model

Random Forest model trained to predict salaries based on job category and location. Model saved for future predictions.